In [2]:
from random import random

import numpy as np
import os
import torch

import utils
from utils import *
from unet import ImCondModel, UNetModel_imcond_wrapper
from topodataset import TopoDataSet
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_otsu, gaussian, sobel
from skimage.morphology import remove_small_holes
from scipy.ndimage import distance_transform_edt
import matplotlib
matplotlib.rcParams.update({"axes.grid" : False})

In [3]:
torch.cuda.is_available()

True

In [4]:
weight_path = '../model_weights/icfm_topo_weights_step_345000.pt'

In [5]:
# load the data:
# DATASETS/DATALOADER
bs=4
num_workers=0
downsampling_factor=1
limit_num_data=np.inf
crop_size=300
use_cfg=True
use_ddc=True
threshold_input=True


dataset = TopoDataSet(
    dir='../data/',
    downsampling_factor=downsampling_factor,
    limit_num_data=limit_num_data,
    crop_size=crop_size,
    threshold_input=threshold_input,
    get_only_last_10p=True
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=bs,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

dataset contains 874 images


In [6]:
# get some images:
all_imgs = []
all_bgs = []
for i, (img, bg) in tqdm(enumerate(dataloader), total=len(dataloader)):
    all_imgs.append(img)
    all_bgs.append(bg)


  0%|          | 0/218 [00:00<?, ?it/s]

In [7]:
all_imgs = torch.cat(all_imgs, dim=0)
all_bgs = torch.cat(all_bgs, dim=0)
all_imgs *= 0.5
all_imgs += 0.5

In [8]:
print(all_imgs.shape)

torch.Size([872, 3, 300, 300])


In [ ]:


# Load the model
h = w = 300
num_channel = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


num_im_cond_channels = 1
if use_cfg:
    num_im_cond_channels  += 1
if use_ddc:
    num_im_cond_channels += 3  # 3 as the cell staining images have 3 channels
model = UNetModel_imcond_wrapper(
    dim=(3, h, w),
    num_res_blocks=2,
    num_channels=num_channel,
    channel_mult=[1, 1, 2, 2, 2],
    num_heads=4,
    num_head_channels=-1,
    attention_ds=(2**5,),
    dropout=0.1,
    num_cond_im_channels=num_im_cond_channels,
).to(device)
    


In [ ]:
checkpoint = torch.load(weight_path, map_location=torch.device(device))

In [ ]:
# checkpoint.keys()

In [ ]:
key = 'ema_model'  # or 'net_model'
# key = 'net_model'
model.load_state_dict(checkpoint[key])

In [ ]:
model.to(device)

# Loading finished -> move to experiments

In [ ]:
import datetime

today = datetime.datetime.today()
def find_center(img, channel=-1):
    # find the max intensity pixerl for the specified channel to determine the center
    i_flat = np.argmax(img[...,channel])
    i = np.unravel_index(i_flat, shape=img[...,channel].shape)
    return i

def zoom_image(img, center, pixels):
    assert len(img.shape) == 3
    low, high = center[0] - pixels//2, center[0] + pixels//2
    low = min(max(0, low), img.shape[0] - pixels)
    high = max(min(img.shape[0], high), pixels)
    left, right = center[1] - pixels//2, center[1] + pixels//2
    left = min(max(0, left), img.shape[0] - pixels)
    right = max(min(img.shape[0], right), pixels)
    img = img[low:high, left:right]
    return img, (low, high, left, right)

In [ ]:
# choose image and observed channel:
idx = 1  # idxs 1, 22
observed_channel = 2 # 0 -- yap, 1 -- actin, 2 -- dapi


channel_names = ['YAP', 'Actin', 'Dapi']
img_and_bg_real = torch.cat([all_bgs[idx], all_imgs[idx]], dim=0)
to_plot = plot_topo_cells(img_and_bg_real.cpu().permute(1, 2, 0).numpy(), bg_norm=0.5) / 255.
plt.imshow(to_plot)
plt.show()

In [ ]:
# generate images conditioned on the observed stain and background of the first image

k = 3
bs = 3

imgs_gen = []
imgs_real = []


for i in tqdm(range(np.ceil(k/bs).astype(int))):
    bg = all_bgs[idx][None, ...].to(device)
    bg = utils.prepare_cfg_conditioning(bg, 0) if use_cfg else bg
    some_bgs = bg[None, ...].repeat(bs,1, 1,1,1).to(device)
    mask = torch.zeros(bs,1, 3, h, w).to(device)
    mask[:,:, observed_channel] = 1  # observed channel
    img = all_imgs[idx]
    thresh = threshold_otsu(img.cpu().numpy())
    img_masked = img #* (img > thresh).float()
    x1 = img_masked.repeat(bs,1,1,1).to(device)*2-1
    imgs_this_gen = generate_samples_topo(model, None, 0, 'test', some_bgs, num_int_steps=100, obs_mask=mask,
                                          x1=x1)
    imgs_gen.append(imgs_this_gen)
    img_and_bg_real = torch.cat([all_bgs[idx][None, ...], all_imgs[idx][None, ...]], dim=1).repeat(bs,1,1,1)
    imgs_real.append(img_and_bg_real)

imgs_gen = torch.cat(imgs_gen, dim=0)
imgs_real = torch.cat(imgs_real, dim=0)



In [ ]:
# # verify that the observed channel remains the same in generated images
# gen = imgs_gen[1,observed_channel+ 1]
# plt.imshow(gen)
# plt.show()

# gen = imgs_gen[2,observed_channel + 1]
# plt.imshow(gen)
# plt.show()

In [ ]:
# plotting parameters:
channels_to_plot_separate = sorted(list(set([0,1,2]) - set([observed_channel])))  # 0 - yap, 1 - actin, 2 - dapi
k_to_plot = 5
pixels = 250
fig, axs = plt.subplots(2, k_to_plot, figsize=(k_to_plot*4,8.5))

spath = 'qualitative_images_partial_obs/'
if not os.path.exists(spath):
    os.makedirs(spath)
    



# now plot the fake completions:
# first plot real image's input channel on the left
# plot fake completions:
for i, channel_to_plot_separate in enumerate(channels_to_plot_separate):
    for j in range(k_to_plot):
        img_and_bg = imgs_real[0, :] if j <=1 else imgs_gen[j-2, :]
        img_and_bg_reshaped = img_and_bg.cpu().permute(1, 2, 0).numpy()
        mask = np.zeros_like(img_and_bg_reshaped)
        ch = observed_channel if j==0 else channel_to_plot_separate
        mask[..., ch+1] = 1   # ch+1 because we still have the bg here
        mask[..., 0] = 1
        im_to_plot =  plot_topo_cells(img_and_bg_reshaped * mask, bg_norm=0.35) / 255.
            
        center = find_center(imgs_real[0, :].cpu().permute(1, 2, 0).numpy()[..., 1:], channel=-1)  # always base on dapi center of real so that loc stays consistent
        im_to_plot, _ = zoom_image(im_to_plot, center, pixels)
        
        axs[i,j].imshow(im_to_plot)
        axs[i,j].axis('off')
        if j == 0:
            lab = f'{channel_names[observed_channel]} (input)'
        elif j == 1:
            lab = f'{channel_names[channel_to_plot_separate]} (real)'
        else:
            lab = f'{channel_names[channel_to_plot_separate]} (generated)'
        
        axs[i,j].set_title(lab, fontsize=16)



    
plt.tight_layout(w_pad=1.08, h_pad=1.08)
# plt.savefig(spath + f'{today.month}_{today.day}_qualitative_obs__{idx}_obs_{channel_names[observed_channel]}.png', dpi=100, transparent=True)
plt.show()